# YOLO26-P2 LLM-GE launch commands

Copy these Bash blocks in order. Mac commands run on the Mac; all other blocks run in the **same PACE terminal** from the repository root. The desired model code and dependencies must already be in that checkout. The environment block appears before the tag check to avoid the import error encountered in this chat. Each new run needs a fresh tag. These commands submitted the server and controller on October 28, 2026; completion and Pareto plots have not been verified here.

## 1. Connect from the Mac

```bash
ssh agupta3057@login-ice.pace.gatech.edu
```

## 2. Enter the repository on PACE

```bash
cd /storage/ice1/6/9/agupta3057/llm-guided-evolution-object-detection
module load uv
```

## 3. Set the YOLO26-P2 environment

```bash
export FRED_SEED_FAMILY=yolo26_p2 FRED_NUM_GENERATIONS=21
export LLMGE_PORT=8479 LLMGE_CLUSTER=pace-ice
export LOCAL=false LOCAL_LLM=true INFERENCE_SUBMISSION=false
export LLMGE_LLM_MODEL=mixtral SUBMIT_ISLAND_CONTROLLER=0
export HF_HUB_DISABLE_XET=1
unset FRED_EVENTS LLMGE_BOUNDED_SESSION LLMGE_MODEL_PATH
unset LLM_MAX_NEW_TOKENS LLMGE_EVAL_NO_PROGRESS_TIMEOUT_SECONDS
unset LLMGE_MIN_SERVER_REMAINING_SECONDS LLMGE_REAL_SQUEUE
```

## 4. Create a new tag

```bash
RUN_DATE=$(date +%Y%m%d)
RUN_SUFFIX=$(date +%H%M%S)
export FRED_RUN_TAG="yolo26_p2_${RUN_DATE}_eot0_trial${RUN_SUFFIX}"
echo "new tag: $FRED_RUN_TAG"
test ! -e "fred_runs/$FRED_RUN_TAG" && echo 'new tag OK'
```

## 5. Confirm the selected configuration

```bash
uv run python -c 'from src.cfg import constants as c; print(c.SEED_FAMILY, c.RUN_TAG, c.SEED_NETWORK, c.num_generations)'
```

Expected: yolo26_p2, the new tag, seed_yolo26n_p2.yaml, and 21.

## 6. Confirm search data and generate Slurm scripts

```bash
test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_train_step50/data.yaml && echo 'training data ready'
test -s sota/FRED_LLM_GE/phase1_detection/seeds/yolo11/data_cache/challenging/search_val_step10/data.yaml && echo 'validation data ready'
mkdir -p run_job_outputs/{server,islands,evolution,evaluation}
uv run python slurm.py
```

Both data-ready messages and a successful slurm.py completion are required before submission.

## 7. Submit the LLM server

```bash
squeue -u "$USER" -o '%i %T %j %R %L'
SERVER_JOB_ID=$(sbatch --parsable --partition=coe-gpu --export=ALL server.sh)
export LLMGE_SERVER_JOB_ID=${SERVER_JOB_ID%%;*}
echo "server job: $LLMGE_SERVER_JOB_ID"
squeue -j "$LLMGE_SERVER_JOB_ID" -o '%i %T %R %L'
```

Wait for RUNNING, then follow the server log:

```bash
tail -F "run_job_outputs/server/slurm-$LLMGE_SERVER_JOB_ID.out"
```

Wait for Uvicorn to report that startup completed. Press Ctrl-C to leave tail; the server job keeps running.

## 8. Submit exactly one controller

```bash
RUN_JOB_ID=$(sbatch --parsable --export=ALL run.sh)
RUN_JOB_ID=${RUN_JOB_ID%%;*}
echo "controller job: $RUN_JOB_ID"
tail -F "run_job_outputs/islands/slurm-$RUN_JOB_ID.out"
```

Press Ctrl-C to leave tail without stopping the job. Record both job IDs and the run tag. Do not submit a second controller for the same tag.